In [17]:
import torch
import tiktoken
import torch.nn as nn
import urllib.request as req
import torch.nn.functional as F

Download Data

In [18]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

In [19]:
try:
    with req.urlopen(url) as response:
        raw_text = response.read().decode('utf-8')
    print("Total Length", len(raw_text))
except:
    print("Text download failed")

Total Length 1115394


In [20]:
corpus = raw_text[:150]
print(corpus)

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

A


Encode Data

In [21]:
enc = tiktoken.encoding_for_model("gpt-4o")
token_ids = enc.encode(raw_text)
vocab_size = enc.max_token_value + 1

In [22]:
data_tensor = torch.tensor(token_ids, dtype=torch.long)

Batach Data

In [23]:
block_size = 128
batch_size = 16
embedding_dim = 64

n = int(0.9 * len(data_tensor))
train_data = data_tensor[:n]
val_data = data_tensor[n:]

def get_batch(device, split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(0, len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

Causal Attention Block

In [24]:
class CausalAttention(nn.Module):
    def __init__(self, dim):
        self.q = nn.Linear(dim, dim, bias=False)
        self.k = nn.Linear(dim, dim, bias=False)
        self.v = nn.Linear(dim, dim, bias=False)
    
    def forward(self, x):
        B, T, C = x.shape
        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        scores = (Q @ K.transpose(-2, -1)) * (C ** -0.5)
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, T, T)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        probs = F.softmax(scores, dim=-1)
        return probs @ V

Transformer Block

In [25]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, sf): #sf - scale factor
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalAttention(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * sf),
            nn.ReLU(),
            nn.Linear(dim * sf, dim)
        )
    
    def forward(self, x):
        x = self.attn(self.ln1(x))
        x = self.ffn(self.ln2(x))
        return x

Simple GPT

In [26]:
class SimpleGPT(nn.Module):
    def __init__(self, vocab_size, dim, sf, max_len): #sf - scale factor for dim
        super().__init__()
        self.vocab_size = vocab_size
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_len, dim)
        self.block = TransformerBlock(dim, sf)
        self.ln = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size)
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_vectors = self.tok_emb(idx)
        pos_vectors = self.pos_emb(torch.arange(T, device=idx.device))

        x = tok_vectors + pos_vectors
        x = self.block(x)
        x = self.ln(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), targets.reshape(-1))
        
        return logits, loss